
# Laboratorio 06: Problemas de Satisfacción de Restricciones (CSP)

**Curso:** Inteligencia Artificial 2026  
**Fecha:** 13 de abril de 2026

En este laboratorio vamos a implementar la solución de Problemas de Satisfacción de Restricciones (CSP), mediante búsqueda DFS con backtracking y con forward checking.



### Librerías utilizadas

En este laboratorio se utilizarán las siguientes librerías:

- **time**: Para medir el tiempo de ejecución de cada algoritmo.
- **copy**: Para realizar copias profundas de estructuras de datos (dominios en forward checking).
- **math**: Para operaciones matemáticas auxiliares (raíz cuadrada en Sudoku).
- **pandas**: Para la presentación de tablas comparativas de resultados.

In [7]:
import time
import copy
import math
import pandas as pd

---
## Ejercicio 1: Implementación de SudokuSolver

Implementar y comparar dos estrategias de búsqueda en profundidad (DFS) para resolver un Sudoku de n × n (n debe ser un cuadrado ≥ 4):

- **Backtracking**: retrocede cuando encuentra una incompatibilidad de restricciones.
- **Forward Checking (Filtering)**: anticipa fallos eliminando valores del dominio de variables no asignadas.

**Estructura del Árbol de Búsqueda:**
- **Estado Raíz**: El tablero inicial con las posiciones fijas.
- **Nodos**: Tableros parcialmente llenos.
- **Hijos**: El resultado de asignar un número válido (1 a n) a la siguiente celda vacía.
- **Hojas**: Un tablero completo (solución) o un tablero donde no hay valores legales para una celda (poda).

**Requerimientos de la clase `SudokuSolver`:**
- `__init__(self, n, board)`: Inicializa el tamaño y el estado del tablero.
- `is_valid(self, row, col, num)`: Verifica si un número puede ir en esa posición.
- `solve_backtracking()`: Implementación de DFS pura con backtracking.
- `solve_filtering()`: Implementación de DFS con Forward Checking.
- `display()`: Imprime el tablero de forma legible.

In [ ]:
# Ejercicio 1: Implementación de SudokuSolver

---
## Ejercicio 2: Resolución de tableros de Sudoku

Resolver los siguientes tableros de Sudoku con ambos métodos (backtracking y forward checking). Para cada caso mostrar la solución encontrada, el número de nodos visitados y el tiempo de ejecución.

**Casos a resolver:**
- Sudoku de 4 × 4
- Sudoku de 9 × 9 nivel fácil
- Sudoku de 9 × 9 nivel extremo
- Sudoku de 16 × 16

In [ ]:
# Ejercicio 2: Resolución de casos de Sudoku + tabla comparativa

---
## Ejercicio 3: Implementación de NQueensSolver

Colocar N reinas en un tablero de ajedrez de N × N de tal manera que ninguna reina amenace a otra. Se usará una **representación de tipo permutaciones**: un arreglo unidimensional `a` de tamaño N donde el índice representa la fila y `a[i]` representa la columna de esa fila donde se coloca la pieza.

In [ ]:
# Ejercicio 3: Implementación de NQueensSolver

---
## Ejercicio 4: Resolución del problema de N-Reinas

Resolver el problema de N-Reinas mediante **backtracking** y **forward checking** para N = 4, 8, 12 y 15.

Para cada caso se mostrará la solución encontrada, el número de nodos visitados y el tiempo de ejecución. Al final se presenta una tabla comparativa entre ambos métodos.

### 4.1 Definición de la clase `NQueensSolver`

La clase encapsula el estado del tablero y los dos algoritmos de búsqueda.

- **`__init__`**: inicializa el tablero como un arreglo de `-1` (sin asignar).
- **`is_valid`**: dado el estado actual de `self.board`, verifica que colocar una reina en `(row, col)` no genere conflictos por columna ni por diagonal.
- **`solve_backtracking`**: DFS pura — asigna, verifica *después*, y retrocede si hay conflicto.
- **`solve_filtering`**: DFS con forward checking — antes de avanzar, elimina del dominio de filas futuras las columnas y diagonales que quedarían atacadas. Si algún dominio queda vacío, poda esa rama de inmediato.
- **`display`**: imprime el tablero con `Q` para reinas y `.` para celdas vacías.

In [2]:
class NQueensSolver:

    def __init__(self, n):
        self.n = n
        self.board = [-1] * n

    def is_valid(self, row, col):
        for r in range(row):
            c = self.board[r]
            if c == col or abs(c - col) == abs(r - row):
                return False
        return True

    def solve_backtracking(self):
        board = [-1] * self.n
        nodes = [0]

        def backtrack(row):
            if row == self.n:
                return True
            for col in range(self.n):
                nodes[0] += 1
                valid = all(
                    board[r] != col and abs(board[r] - col) != abs(r - row)
                    for r in range(row)
                )
                if valid:
                    board[row] = col
                    if backtrack(row + 1):
                        return True
                    board[row] = -1
            return False

        start = time.time()
        backtrack(0)
        elapsed = time.time() - start
        self.board = board
        return board[:], nodes[0], elapsed

    def solve_filtering(self):
        board = [-1] * self.n
        domains = [set(range(self.n)) for _ in range(self.n)]
        nodes = [0]

        def propagate(row, col, domains):
            new_domains = [d.copy() for d in domains]
            for future_row in range(row + 1, self.n):
                dist = future_row - row
                new_domains[future_row].discard(col)
                new_domains[future_row].discard(col - dist)
                new_domains[future_row].discard(col + dist)
                if not new_domains[future_row]:
                    return None
            return new_domains

        def dfs(row, domains):
            if row == self.n:
                return True
            for col in sorted(domains[row]):
                nodes[0] += 1
                new_domains = propagate(row, col, domains)
                if new_domains is not None:
                    board[row] = col
                    if dfs(row + 1, new_domains):
                        return True
                    board[row] = -1
            return False

        start = time.time()
        dfs(0, domains)
        elapsed = time.time() - start
        self.board = board
        return board[:], nodes[0], elapsed

    def display(self):
        sep = "+" + "-" * (2 * self.n - 1) + "+"
        print(sep)
        for row in range(self.n):
            print("|" + " ".join("Q" if self.board[row] == col else "." for col in range(self.n)) + "|")
        print(sep)

### 4.2 Backtracking — N = 4, 8, 12, 15

DFS pura: explora cada columna fila por fila y retrocede solo cuando detecta un conflicto en el estado actual.

In [3]:
casos = [4, 8, 12, 15]
resultados_bt = []

for n in casos:
    solver = NQueensSolver(n)
    sol, nodos, tiempo = solver.solve_backtracking()

    print(f"── N = {n} ──────────────────────────────")
    print(f"  Solución : {sol}")
    print(f"  Nodos    : {nodos}")
    print(f"  Tiempo   : {tiempo:.6f} s")
    solver.display()
    print()

    resultados_bt.append({"N": n, "Nodos BT": nodos, "Tiempo BT (s)": round(tiempo, 6)})

── N = 4 ──────────────────────────────
  Solución : [1, 3, 0, 2]
  Nodos    : 26
  Tiempo   : 0.000000 s
+-------+
|. Q . .|
|. . . Q|
|Q . . .|
|. . Q .|
+-------+

── N = 8 ──────────────────────────────
  Solución : [0, 4, 7, 5, 2, 6, 1, 3]
  Nodos    : 876
  Tiempo   : 0.003389 s
+---------------+
|Q . . . . . . .|
|. . . . Q . . .|
|. . . . . . . Q|
|. . . . . Q . .|
|. . Q . . . . .|
|. . . . . . Q .|
|. Q . . . . . .|
|. . . Q . . . .|
+---------------+

── N = 12 ──────────────────────────────
  Solución : [0, 2, 4, 7, 9, 11, 5, 10, 1, 6, 8, 3]
  Nodos    : 3066
  Tiempo   : 0.011303 s
+-----------------------+
|Q . . . . . . . . . . .|
|. . Q . . . . . . . . .|
|. . . . Q . . . . . . .|
|. . . . . . . Q . . . .|
|. . . . . . . . . Q . .|
|. . . . . . . . . . . Q|
|. . . . . Q . . . . . .|
|. . . . . . . . . . Q .|
|. Q . . . . . . . . . .|
|. . . . . . Q . . . . .|
|. . . . . . . . Q . . .|
|. . . Q . . . . . . . .|
+-----------------------+

── N = 15 ───────────────────────

### 4.3 Forward Checking — N = 4, 8, 12, 15

Antes de asignar cada columna, se propaga la restricción hacia las filas futuras: se eliminan del dominio las columnas directas y diagonales que quedarían atacadas. Si algún dominio queda vacío, la rama se poda sin explorarla.

In [4]:
resultados_fc = []

for n in casos:
    solver = NQueensSolver(n)
    sol, nodos, tiempo = solver.solve_filtering()

    print(f"── N = {n} ──────────────────────────────")
    print(f"  Solución : {sol}")
    print(f"  Nodos    : {nodos}")
    print(f"  Tiempo   : {tiempo:.6f} s")
    solver.display()
    print()

    resultados_fc.append({"N": n, "Nodos FC": nodos, "Tiempo FC (s)": round(tiempo, 6)})

── N = 4 ──────────────────────────────
  Solución : [1, 3, 0, 2]
  Nodos    : 8
  Tiempo   : 0.000000 s
+-------+
|. Q . .|
|. . . Q|
|Q . . .|
|. . Q .|
+-------+

── N = 8 ──────────────────────────────
  Solución : [0, 4, 7, 5, 2, 6, 1, 3]
  Nodos    : 88
  Tiempo   : 0.000000 s
+---------------+
|Q . . . . . . .|
|. . . . Q . . .|
|. . . . . . . Q|
|. . . . . Q . .|
|. . Q . . . . .|
|. . . . . . Q .|
|. Q . . . . . .|
|. . . Q . . . .|
+---------------+

── N = 12 ──────────────────────────────
  Solución : [0, 2, 4, 7, 9, 11, 5, 10, 1, 6, 8, 3]
  Nodos    : 193
  Tiempo   : 0.001310 s
+-----------------------+
|Q . . . . . . . . . . .|
|. . Q . . . . . . . . .|
|. . . . Q . . . . . . .|
|. . . . . . . Q . . . .|
|. . . . . . . . . Q . .|
|. . . . . . . . . . . Q|
|. . . . . Q . . . . . .|
|. . . . . . . . . . Q .|
|. Q . . . . . . . . . .|
|. . . . . . Q . . . . .|
|. . . . . . . . Q . . .|
|. . . Q . . . . . . . .|
+-----------------------+

── N = 15 ──────────────────────────

### 4.4 Tabla comparativa

In [5]:
df_bt = pd.DataFrame(resultados_bt)
df_fc = pd.DataFrame(resultados_fc)
df = df_bt.merge(df_fc, on="N")

df["Reducción nodos (%)"] = ((1 - df["Nodos FC"] / df["Nodos BT"]) * 100).round(2)
df["Speedup (x)"] = (df["Tiempo BT (s)"] / df["Tiempo FC (s)"]).round(2)

print(df.to_string(index=False))

 N  Nodos BT  Tiempo BT (s)  Nodos FC  Tiempo FC (s)  Reducción nodos (%)  Speedup (x)
 4        26       0.000000         8       0.000000                69.23          NaN
 8       876       0.003389        88       0.000000                89.95          inf
12      3066       0.011303       193       0.001310                93.71         8.63
15     20280       0.055504      1026       0.002434                94.94        22.80


### 4.5 Análisis de resultados

A partir de la tabla comparativa se puede observar que el **Forward Checking** es consistentemente más eficiente que el Backtracking puro en todos los casos evaluados:

- **N = 4**: La diferencia es pequeña; el espacio de búsqueda es mínimo y ambos algoritmos terminan de forma casi instantánea.
- **N = 8**: El Forward Checking comienza a mostrar una reducción notable de nodos, ya que anticipa y elimina ramas inviables antes de explorarlas.
- **N = 12 y N = 15**: La ventaja se vuelve muy significativa. La reducción de nodos supera el 90%, lo que se traduce directamente en menor tiempo de ejecución.

La **reducción porcentual de nodos visitados** crece con N, confirmando que el Forward Checking escala mejor ante instancias más grandes del problema.

---
## Ejercicio 5: Todas las soluciones para N-Reinas

Modificar el algoritmo de backtracking para obtener **todas** las soluciones del problema de N-Reinas para N = 4, 5 y 6.

La modificación clave respecto a `solve_backtracking` es que al llegar al caso base **no se retorna `True`**: en su lugar se guarda la solución en una lista y se continúa explorando, deshaciendo siempre la última asignación para seguir probando otras columnas.

### 5.1 Definición de `NQueensAllSolutions`

In [10]:
class NQueensAllSolutions:

    def __init__(self, n):
        self.n = n
        self.solutions = []

    def _is_valid(self, board, row, col):
        for r in range(row):
            c = board[r]
            if c == col or abs(c - col) == abs(r - row):
                return False
        return True

    def solve_all(self):
        board = [-1] * self.n
        self.solutions = []
        nodes = [0]

        def backtrack(row):
            if row == self.n:
                self.solutions.append(board[:])
                return
            for col in range(self.n):
                nodes[0] += 1
                if self._is_valid(board, row, col):
                    board[row] = col
                    backtrack(row + 1)
                    board[row] = -1

        start = time.time()
        backtrack(0)
        elapsed = time.time() - start
        return self.solutions, nodes[0], elapsed

    def display_solution(self, solution):
        sep = "+" + "-" * (2 * self.n - 1) + "+"
        print(sep)
        for row in range(self.n):
            print("|" + " ".join("Q" if solution[row] == col else "." for col in range(self.n)) + "|")
        print(sep)

### 5.2 Resolución para N = 4, 5, 6

In [11]:
for n in [4, 5, 6]:
    solver = NQueensAllSolutions(n)
    soluciones, nodos, tiempo = solver.solve_all()

    print(f"══ N = {n} ══════════════════════════════")
    print(f"  Total soluciones : {len(soluciones)}")
    print(f"  Nodos visitados  : {nodos}")
    print(f"  Tiempo           : {tiempo:.6f} s\n")

    for idx, sol in enumerate(soluciones, 1):
        print(f"  Solución {idx}: {sol}")
        solver.display_solution(sol)
        print()

══ N = 4 ══════════════════════════════
  Total soluciones : 2
  Nodos visitados  : 60
  Tiempo           : 0.000413 s

  Solución 1: [1, 3, 0, 2]
+-------+
|. Q . .|
|. . . Q|
|Q . . .|
|. . Q .|
+-------+

  Solución 2: [2, 0, 3, 1]
+-------+
|. . Q .|
|Q . . .|
|. . . Q|
|. Q . .|
+-------+

══ N = 5 ══════════════════════════════
  Total soluciones : 10
  Nodos visitados  : 220
  Tiempo           : 0.000000 s

  Solución 1: [0, 2, 4, 1, 3]
+---------+
|Q . . . .|
|. . Q . .|
|. . . . Q|
|. Q . . .|
|. . . Q .|
+---------+

  Solución 2: [0, 3, 1, 4, 2]
+---------+
|Q . . . .|
|. . . Q .|
|. Q . . .|
|. . . . Q|
|. . Q . .|
+---------+

  Solución 3: [1, 3, 0, 2, 4]
+---------+
|. Q . . .|
|. . . Q .|
|Q . . . .|
|. . Q . .|
|. . . . Q|
+---------+

  Solución 4: [1, 4, 2, 0, 3]
+---------+
|. Q . . .|
|. . . . Q|
|. . Q . .|
|Q . . . .|
|. . . Q .|
+---------+

  Solución 5: [2, 0, 3, 1, 4]
+---------+
|. . Q . .|
|Q . . . .|
|. . . Q .|
|. Q . . .|
|. . . . Q|
+---------+

  Soluc

### 5.3 Resumen de conteo

In [12]:
resumen = []
for n in [4, 5, 6]:
    solver = NQueensAllSolutions(n)
    soluciones, nodos, tiempo = solver.solve_all()
    resumen.append({"N": n, "Total soluciones": len(soluciones), "Nodos visitados": nodos, "Tiempo (s)": round(tiempo, 6)})

print(pd.DataFrame(resumen).to_string(index=False))

 N  Total soluciones  Nodos visitados  Tiempo (s)
 4                 2               60    0.000000
 5                10              220    0.000000
 6                 4              894    0.001632


### 5.4 ¿Cuántas soluciones hay en cada caso?

Los resultados obtenidos coinciden con los valores teóricos conocidos para el problema de N-Reinas:

| N | Soluciones |
|---|------------|
| 4 | 2          |
| 5 | 10         |
| 6 | 4          |

- **N = 4**: Solo 2 configuraciones son válidas. El tablero es tan pequeño que las restricciones diagonales dejan muy poco margen.
- **N = 5**: El número sube a 10, ya que el tablero más grande permite mayor variedad de configuraciones sin conflictos.
- **N = 6**: Baja a 4 a pesar del tablero más grande — las diagonales resultan más restrictivas en proporción al tamaño.